# Qwen2-VL-2B QLoRA 파인튜닝 평가

## 평가 목표
1. **파인튜닝 전/후 정량 비교** — BLEU-4, ROUGE-L, 키워드 정확도
2. **QA 유형별 성능** — 7가지 유형별 강약점 파악
3. **오류 사례 분석** — 어떤 케이스를 틀리는지 시각화
4. **자유 질문 테스트** — 학습에 없던 질문으로 실전 감각 확인

## 핵심 질문
- Base 모델이 "이미지 분석 불가"라고 했던 질문에 파인튜닝 모델은 무엇을 답하는가?
- 거리 예측 오차는 얼마나 되는가?
- 어떤 유형이 가장 약한가?

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import json
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict

print('라이브러리 로드 완료')

## 셀 1: 파인튜닝 전/후 비교 결과 분석 (저장된 JSON 활용)

> **finetune_comparison.json** 에 이미 before/after 샘플이 있습니다.
> 모델을 다시 로드하지 않고 저장된 결과로 먼저 분석합니다.

In [ ]:
# ─── 저장된 비교 결과 로드 ───────────────────────────────
comparison_path = Path('finetune_comparison.json')
with open(comparison_path, encoding='utf-8') as f:
    comparisons = json.load(f)

print(f'비교 샘플 수: {len(comparisons)}개')
print()

# 샘플 출력
for i, c in enumerate(comparisons):
    print(f'[샘플 {i+1}] {c["qa_type"]}')
    print(f'  질문  : {c["question"]}')
    print(f'  정답  : {c["gt"]}')
    print(f'  ─ Before: {c["before"][:80]}...' if len(c['before']) > 80 else f'  ─ Before: {c["before"]}')
    print(f'  ─ After : {c["after"]}')
    print()

## 셀 2: BLEU / ROUGE-L 정량 비교

BLEU = 단어 n-gram 겹침 (얼마나 비슷한 표현을 썼나)
ROUGE-L = 가장 긴 공통 부분 수열 (문장 흐름 유사도)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# nltk 없으면 설치
try:
    import nltk
    nltk.data.find('tokenizers/punkt')
except:
    import nltk
    nltk.download('punkt', quiet=True)

smooth = SmoothingFunction().method1

def char_bleu(ref, hyp):
    """한국어는 공백 기준 토크나이저 사용"""
    ref_tokens = ref.split()
    hyp_tokens = hyp.split()
    if not hyp_tokens:
        return 0.0
    return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth)

def _lcs_len(ref_tokens: list, hyp_tokens: list) -> int:
    """LCS 길이 — DP로 계산 (두 행만 유지)"""
    m, n = len(ref_tokens), len(hyp_tokens)
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                curr[j] = prev[j-1] + 1
            else:
                curr[j] = max(prev[j], curr[j-1])
        prev = curr
    return prev[n]

def rouge_l(ref: str, hyp: str) -> float:
    """한국어 ROUGE-L (공백 기준 토큰, LCS F1)

    rouge_score 라이브러리는 영어 기준 토크나이저를 써서
    한국어 문장에서 0이 나오는 버그가 있음 → 직접 구현
    """
    ref_tokens = ref.split()
    hyp_tokens = hyp.split()
    if not ref_tokens or not hyp_tokens:
        return 0.0
    lcs = _lcs_len(ref_tokens, hyp_tokens)
    precision = lcs / len(hyp_tokens)
    recall    = lcs / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# ─── 검증 ─────────────────────────────────────────────────
# 거의 같은 문장 → 0.8+ 나와야 정상
_test_gt   = '가장 가까운 객체는 좌측에 있는 보행자이며, 약 16.1m 거리에 있습니다.'
_test_pred = '가장 가까운 객체는 좌측에 있는 보행자이며, 약 30.3m 거리에 있습니다.'
_score = rouge_l(_test_gt, _test_pred)
print(f'ROUGE-L 검증 (거의 같은 문장): {_score:.4f}  {"✓ 정상" if _score > 0.5 else "✗ 이상"}')

# before/after 계산
results = []
for c in comparisons:
    gt = c['gt']
    before = c['before']
    after = c['after']
    results.append({
        'qa_type': c['qa_type'],
        'bleu_before': char_bleu(gt, before),
        'bleu_after':  char_bleu(gt, after),
        'rouge_before': rouge_l(gt, before),
        'rouge_after':  rouge_l(gt, after),
    })

avg_bleu_before  = np.mean([r['bleu_before']  for r in results])
avg_bleu_after   = np.mean([r['bleu_after']   for r in results])
avg_rouge_before = np.mean([r['rouge_before'] for r in results])
avg_rouge_after  = np.mean([r['rouge_after']  for r in results])

print()
print('=' * 50)
print(f'  지표           Before     After    향상')
print('=' * 50)
print(f'  BLEU          {avg_bleu_before:.4f}    {avg_bleu_after:.4f}   +{avg_bleu_after - avg_bleu_before:.4f}')
print(f'  ROUGE-L       {avg_rouge_before:.4f}    {avg_rouge_after:.4f}   +{avg_rouge_after - avg_rouge_before:.4f}')
print('=' * 50)

## 셀 3: Val 셋 전체 평가 (파인튜닝 모델 로드)

모델을 로드해서 val.json 전체를 추론합니다. GPU 메모리 절약을 위해 4bit 양자화 사용.

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from PIL import Image

BASE_MODEL = 'Qwen/Qwen2-VL-2B-Instruct'
ADAPTER_PATH = 'lora_adapter'

print('모델 로드 중... (2~3분 소요)')

# 4bit 양자화로 메모리 절약
from transformers import BitsAndBytesConfig
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

processor = AutoProcessor.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
print('모델 로드 완료!')

In [ ]:
# ─── 추론 함수 ────────────────────────────────────────────
@torch.inference_mode()
def predict(image_path: str, question: str, max_new_tokens: int = 128) -> str:
    """이미지 + 질문 → 답변 생성"""
    img = Image.open(image_path).convert('RGB')
    
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': question},
        ]
    }]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors='pt',
    ).to(model.device)
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
    )
    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

print('추론 함수 준비 완료')

In [ ]:
# ─── Val 셋 전체 추론 (402개) ────────────────────────────
# 시간이 걸립니다 (RTX 4080 기준 약 10~20분)
# 빠른 확인이 필요하면 SAMPLE_SIZE를 줄이세요

VAL_JSON = Path('qa_dataset/val.json')
SAMPLE_SIZE = 100  # None이면 전체 402개

with open(VAL_JSON, encoding='utf-8') as f:
    val_data = json.load(f)

if SAMPLE_SIZE:
    # 유형별로 균등하게 샘플링
    from collections import defaultdict
    by_type = defaultdict(list)
    for item in val_data:
        by_type[item['metadata']['qa_type']].append(item)
    
    sampled = []
    per_type = SAMPLE_SIZE // len(by_type)
    for items in by_type.values():
        sampled.extend(items[:per_type])
    val_data = sampled

print(f'평가 샘플 수: {len(val_data)}개')

# ─── 추론 실행 ───────────────────────────────────────────
eval_results = []

for i, item in enumerate(val_data):
    if i % 20 == 0:
        print(f'  진행: {i}/{len(val_data)}')
    
    image_path = item['image']
    question   = item['conversations'][0]['value'].replace('<image>\n', '')
    gt         = item['conversations'][1]['value']
    qa_type    = item['metadata']['qa_type']
    
    try:
        pred = predict(image_path, question)
    except Exception as e:
        pred = f'[ERROR: {e}]'
    
    eval_results.append({
        'qa_type': qa_type,
        'question': question,
        'gt': gt,
        'pred': pred,
        'bleu':  char_bleu(gt, pred),
        'rouge': rouge_l(gt, pred),
    })

print(f'\n추론 완료! 총 {len(eval_results)}개')

# 결과 저장
with open('eval_results.json', 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)
print('eval_results.json 저장 완료')

## 셀 4: QA 유형별 성능 시각화

In [ ]:
# ─── 기존 eval_results.json의 rouge 점수 재계산 ──────────
# rouge_score 라이브러리 버그로 0이 나왔던 것을 고친 함수로 다시 계산

with open('eval_results.json', encoding='utf-8') as f:
    eval_results = json.load(f)

for r in eval_results:
    r['rouge'] = rouge_l(r['gt'], r['pred'])
    r['bleu']  = char_bleu(r['gt'], r['pred'])

with open('eval_results.json', 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

# 확인
all_rouge = [r['rouge'] for r in eval_results]
print(f'재계산 완료: {len(eval_results)}개')
print(f'ROUGE-L 평균: {np.mean(all_rouge):.4f}')
print(f'ROUGE-L 0인 샘플: {sum(1 for x in all_rouge if x == 0)}개')

# 샘플 확인 — 이제 비슷한 문장은 높은 점수가 나와야 함
sample = eval_results[0]
print(f'\n[샘플 확인]')
print(f'  정답: {sample["gt"]}')
print(f'  예측: {sample["pred"]}')
print(f'  ROUGE-L: {sample["rouge"]:.4f}')

In [ ]:
# ─── eval_results.json 로드 (위 셀 실행 후) ─────────────
with open('eval_results.json', encoding='utf-8') as f:
    eval_results = json.load(f)

# ─── 유형별 평균 계산 ─────────────────────────────────────
type_scores = defaultdict(lambda: {'bleu': [], 'rouge': []})

for r in eval_results:
    t = r['qa_type']
    type_scores[t]['bleu'].append(r['bleu'])
    type_scores[t]['rouge'].append(r['rouge'])

types = sorted(type_scores.keys())
avg_bleu  = [np.mean(type_scores[t]['bleu'])  for t in types]
avg_rouge = [np.mean(type_scores[t]['rouge']) for t in types]

# ─── 시각화 ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('QA 유형별 파인튜닝 모델 성능', fontsize=14, fontweight='bold')

x = np.arange(len(types))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(types)))

# BLEU
bars = axes[0].bar(x, avg_bleu, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('BLEU Score (높을수록 좋음)', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(types, rotation=30, ha='right', fontsize=9)
axes[0].set_ylim(0, 1)
axes[0].axhline(y=np.mean(avg_bleu), color='red', linestyle='--', linewidth=1, label=f'평균: {np.mean(avg_bleu):.3f}')
axes[0].legend(fontsize=10)
for bar, val in zip(bars, avg_bleu):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# ROUGE-L
bars2 = axes[1].bar(x, avg_rouge, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('ROUGE-L Score (높을수록 좋음)', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(types, rotation=30, ha='right', fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].axhline(y=np.mean(avg_rouge), color='red', linestyle='--', linewidth=1, label=f'평균: {np.mean(avg_rouge):.3f}')
axes[1].legend(fontsize=10)
for bar, val in zip(bars2, avg_rouge):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('eval_scores_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── 수치 요약 ────────────────────────────────────────────
print('\n' + '=' * 55)
print(f'{"QA 유형":<25}  {"BLEU":>8}  {"ROUGE-L":>8}  {"샘플수":>5}')
print('=' * 55)
for t in types:
    b = np.mean(type_scores[t]['bleu'])
    r = np.mean(type_scores[t]['rouge'])
    n = len(type_scores[t]['bleu'])
    print(f'{t:<25}  {b:>8.4f}  {r:>8.4f}  {n:>5}')
print('=' * 55)
print(f'{"전체 평균":<25}  {np.mean(avg_bleu):>8.4f}  {np.mean(avg_rouge):>8.4f}')

## 셀 5: 거리 예측 오차 분석

장면설명/nearest_object 질문에서 거리 수치(m)를 추출해서 정량 비교

In [ ]:
def extract_distances(text: str) -> list[float]:
    """텍스트에서 숫자m 패턴 추출 (예: '30.6m' → 30.6)"""
    pattern = r'(\d+\.?\d*)m'
    return [float(x) for x in re.findall(pattern, text)]

# 거리 관련 유형만 필터
distance_types = {'scene_description', 'nearest_object', 'danger_assessment'}
distance_data = [r for r in eval_results if r['qa_type'] in distance_types]

errors = []
for r in distance_data:
    gt_dists   = extract_distances(r['gt'])
    pred_dists = extract_distances(r['pred'])
    
    if gt_dists and pred_dists:
        # 가장 가까운 거리 기준으로 비교
        gt_min   = min(gt_dists)
        pred_min = min(pred_dists)
        errors.append(abs(gt_min - pred_min))

if errors:
    errors = np.array(errors)
    print('거리 예측 오차 분석 (거리 언급이 있는 샘플만)')
    print(f'  샘플 수  : {len(errors)}개')
    print(f'  MAE      : {errors.mean():.2f}m  (평균 절대 오차)')
    print(f'  중앙값   : {np.median(errors):.2f}m')
    print(f'  최대오차 : {errors.max():.2f}m')
    print(f'  5m 이내  : {(errors < 5).mean()*100:.1f}%')
    print(f'  10m 이내 : {(errors < 10).mean()*100:.1f}%')
    
    # 오차 분포 시각화
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(errors, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(errors.mean(), color='red', linestyle='--', label=f'MAE = {errors.mean():.2f}m')
    ax.set_xlabel('거리 예측 오차 (m)', fontsize=11)
    ax.set_ylabel('샘플 수', fontsize=11)
    ax.set_title('거리 예측 오차 분포', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('distance_error_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('거리 데이터가 충분하지 않습니다.')

## 셀 6: 오류 사례 심층 분석

ROUGE-L이 낮은 (= 가장 많이 틀린) 샘플을 뽑아서 원인 파악

In [ ]:
# ─── ROUGE-L 기준 최하위 10개 ────────────────────────────
worst = sorted(eval_results, key=lambda x: x['rouge'])[:10]

print('■ 가장 틀린 10개 샘플')
print('=' * 70)
for i, r in enumerate(worst):
    print(f'\n[{i+1}] 유형: {r["qa_type"]}  |  ROUGE-L: {r["rouge"]:.4f}')
    print(f'  질문: {r["question"]}')
    print(f'  정답: {r["gt"]}')
    print(f'  예측: {r["pred"]}')

In [ ]:
# ─── 오류 유형 분류 ───────────────────────────────────────
# 오류 원인 패턴 분석
error_patterns = {
    '거리 방향 오류': 0,    # 좌/우/전방 틀림
    '거리 수치 오류': 0,    # 거리가 크게 다름
    '객체 유형 오류': 0,    # 차량/보행자 혼동
    '거리 누락':     0,    # 거리 언급 없음
    '기타':          0,
}

direction_words = ['좌측', '우측', '전방', '후방']
object_words    = ['차량', '보행자', '자전거', '신호등']

for r in worst:
    gt, pred = r['gt'], r['pred']
    
    # 방향 체크
    gt_dirs   = [w for w in direction_words if w in gt]
    pred_dirs = [w for w in direction_words if w in pred]
    if gt_dirs and pred_dirs and set(gt_dirs) != set(pred_dirs):
        error_patterns['거리 방향 오류'] += 1
        continue
    
    # 거리 체크
    gt_d   = extract_distances(gt)
    pred_d = extract_distances(pred)
    if gt_d and not pred_d:
        error_patterns['거리 누락'] += 1
        continue
    if gt_d and pred_d and abs(min(gt_d) - min(pred_d)) > 15:
        error_patterns['거리 수치 오류'] += 1
        continue
    
    # 객체 체크
    gt_obj   = [w for w in object_words if w in gt]
    pred_obj = [w for w in object_words if w in pred]
    if gt_obj and pred_obj and set(gt_obj) != set(pred_obj):
        error_patterns['객체 유형 오류'] += 1
        continue
    
    error_patterns['기타'] += 1

print('\n오류 패턴 분석:')
for pattern, count in error_patterns.items():
    bar = '█' * count
    print(f'  {pattern:<12}: {bar} ({count})')

## 셀 7: 자유 질문 테스트 (실전 감각)

학습 데이터에 없던 새로운 이미지 + 자유 질문으로 실전 성능 확인

In [ ]:
import glob

# ─── 테스트 이미지 선택 ───────────────────────────────────
# val 셋에서 사용하지 않은 이미지 찾기
all_images = sorted(glob.glob(
    'phase4_carla/data_collection/carla_dataset/images/*.jpg'
))

val_images = set(r['gt'] for r in eval_results)  # 이미 평가한 이미지
# 마지막 3장을 자유 테스트용으로 사용
test_images = all_images[-3:] if all_images else []

print(f'전체 이미지: {len(all_images)}장, 테스트: {len(test_images)}장')

# ─── 자유 질문 리스트 ─────────────────────────────────────
free_questions = [
    '현재 주행 속도를 줄여야 하나요? 이유는?',
    '지금 상황에서 좌회전해도 안전한가요?',
    '이 장면의 위험 등급을 1~5로 매겨주세요.',
    '주변 보행자가 몇 명이고, 가장 위험한 보행자는 어디 있나요?',
]

if test_images:
    test_img = test_images[0]
    print(f'\n테스트 이미지: {Path(test_img).name}')
    print('=' * 60)
    
    for q in free_questions:
        pred = predict(test_img, q)
        print(f'\n❓ {q}')
        print(f'→  {pred}')
    
    # 이미지 표시
    img = Image.open(test_img)
    plt.figure(figsize=(8, 4))
    plt.imshow(img)
    plt.title(f'테스트 이미지: {Path(test_img).name}', fontsize=11)
    plt.axis('off')
    plt.show()
else:
    print('이미지를 찾을 수 없습니다.')

## 셀 8: 최종 성능 요약 리포트

In [ ]:
# ─── 최종 요약 시각화 ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Qwen2-VL-2B QLoRA 파인튜닝 최종 평가', fontsize=14, fontweight='bold')

# 1. Before vs After (finetune_comparison.json 기반)
metrics = ['BLEU', 'ROUGE-L']
before_vals = [avg_bleu_before, avg_rouge_before]
after_vals  = [avg_bleu_after, avg_rouge_after]

x_pos = np.arange(len(metrics))
w = 0.35
axes[0].bar(x_pos - w/2, before_vals, w, label='Base 모델', color='#ff6b6b', alpha=0.85)
axes[0].bar(x_pos + w/2, after_vals,  w, label='파인튜닝', color='#51cf66', alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(metrics, fontsize=11)
axes[0].set_ylim(0, 1)
axes[0].set_title('파인튜닝 전/후 비교', fontsize=12)
axes[0].legend(fontsize=10)
for i, (b, a) in enumerate(zip(before_vals, after_vals)):
    axes[0].text(i - w/2, b + 0.02, f'{b:.3f}', ha='center', fontsize=9, color='#c92a2a')
    axes[0].text(i + w/2, a + 0.02, f'{a:.3f}', ha='center', fontsize=9, color='#2f9e44')

# 2. 유형별 ROUGE-L
type_rouge = [(t, np.mean(type_scores[t]['rouge'])) for t in types]
type_rouge.sort(key=lambda x: x[1], reverse=True)
t_labels, t_vals = zip(*type_rouge)
colors2 = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(t_labels)))
axes[1].barh(range(len(t_labels)), t_vals, color=colors2[::-1])
axes[1].set_yticks(range(len(t_labels)))
axes[1].set_yticklabels(t_labels[::-1], fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_title('유형별 ROUGE-L (높을수록 좋음)', fontsize=12)
axes[1].axvline(np.mean(t_vals), color='red', linestyle='--', linewidth=1)

# 3. 학습 곡선 (training_args.bin 대신 trainer_state.json 활용)
trainer_state_path = Path('checkpoints/checkpoint-843/trainer_state.json')
if trainer_state_path.exists():
    with open(trainer_state_path) as f:
        state = json.load(f)
    
    log_history = state.get('log_history', [])
    train_loss = [(l['step'], l['loss']) for l in log_history if 'loss' in l]
    val_loss   = [(l['step'], l['eval_loss']) for l in log_history if 'eval_loss' in l]
    
    if train_loss:
        steps, losses = zip(*train_loss)
        axes[2].plot(steps, losses, label='Train Loss', color='steelblue', alpha=0.7)
    if val_loss:
        vsteps, vlosses = zip(*val_loss)
        axes[2].plot(vsteps, vlosses, 'o-', label='Val Loss', color='tomato', linewidth=2)
    axes[2].set_title('학습 곡선', fontsize=12)
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Loss')
    axes[2].legend(fontsize=10)
    axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'trainer_state.json\n없음', ha='center', va='center', fontsize=12)
    axes[2].axis('off')

plt.tight_layout()
plt.savefig('eval_final_report.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── 텍스트 요약 ─────────────────────────────────────────
print('\n' + '=' * 60)
print('  Qwen2-VL-2B QLoRA 파인튜닝 — 최종 성능 요약')
print('=' * 60)
print(f'  [파인튜닝 효과]')
print(f'    BLEU   : {avg_bleu_before:.4f} → {avg_bleu_after:.4f}  (+{avg_bleu_after - avg_bleu_before:.4f})')
print(f'    ROUGE-L: {avg_rouge_before:.4f} → {avg_rouge_after:.4f}  (+{avg_rouge_after - avg_rouge_before:.4f})')
print()
print(f'  [Val 셋 성능 ({len(eval_results)}개)]')
print(f'    BLEU    평균: {np.mean([r["bleu"]  for r in eval_results]):.4f}')
print(f'    ROUGE-L 평균: {np.mean([r["rouge"] for r in eval_results]):.4f}')
print()
if errors is not None and len(errors) > 0:
    print(f'  [거리 예측 오차]')
    print(f'    MAE: {errors.mean():.2f}m  (5m 이내: {(errors < 5).mean()*100:.1f}%)')
print()
best_type  = max(type_rouge, key=lambda x: x[1])
worst_type = min(type_rouge, key=lambda x: x[1])
print(f'  [강점/약점]')
print(f'    최고 유형: {best_type[0]}  (ROUGE-L {best_type[1]:.4f})')
print(f'    최저 유형: {worst_type[0]}  (ROUGE-L {worst_type[1]:.4f})')
print('=' * 60)